# Módulo 03 · Aula 2: RAG corretivo

**Duração:** 1 h

## Objetivos de aprendizagem

- Reconhecer por que um RAG "em linha reta" não percebe quando a busca falhou.
- Avaliar a relevância dos trechos com um nó `grade` e saída estruturada (`RelevanceGrade`).
- Decidir o próximo passo com uma aresta condicional (`route`): gerar, reescrever a consulta ou desistir.
- Reescrever a consulta com o `REWRITE_PROMPT`, com um limite de tentativas (`MAX_REWRITES`), e responder com segurança
  (`no_answer`) quando o limite acaba.
- Usar `build_rag_graph` do produto, acompanhar cada passo com `stream_mode="updates"` e medir o custo de cada caminho.

## Pré-requisitos

- Aula 3.1: estado do RAG, conversa com checkpointer e o nó `condense`.
- Curso de LangGraph: arestas condicionais, ciclos com limite e streaming.

## O que vamos construir

As peças do RAG corretivo, uma de cada vez, e depois o grafo completo do produto, em
[src/graph/corrective_rag.py](../src/graph/corrective_rag.py):

```mermaid
flowchart LR
    S([START]) --> C[condense] --> R[retrieve] --> G[grade<br/>LLM]
    G -.há trechos relevantes.-> GE[generate<br/>LLM] --> E([END])
    G -.nenhum relevante, ainda há tentativas.-> RW[rewrite<br/>LLM] --> R
    G -.nenhum relevante, limite atingido.-> NA[no_answer] --> E
```

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

In [2]:
import logging

from src.rag.retrieval import HybridRetriever
from src.rag.vectorstore import index_exists, load_index

if not index_exists():
    from src.rag.ingestion import index_folder

    index_folder()

retriever = HybridRetriever(index=load_index())
# Os logs do produto repetem o que o stream vai mostrar; deixamos só avisos e erros.
logging.getLogger("src").setLevel(logging.WARNING)


def origin(chunk) -> str:
    """Origem curta de um trecho: arquivo e, nos PDFs, a página."""
    md = chunk.metadata
    return md["source"] + (f" p.{md['page']}" if md.get("page") else "")

2026-09-27 20:55:23 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


## 1. Quando a busca falha, o RAG simples não percebe

Um retriever **sempre** devolve `k` trechos: a similaridade é relativa, então "os mais parecidos" existem mesmo quando
nada na base responde à pergunta. Veja o que volta para uma pergunta que a base não cobre:

In [3]:
job_question = "A TechNova está contratando? Onde mando meu currículo?"
for chunk in retriever.invoke(job_question):
    print(f"{origin(chunk):<45} | {chunk.page_content[:80]!r}")

help_center/support_channels_and_hours.html   | '[Canais e horários de atendimento | Central de Ajuda TechNova]\nOutros contatos\n\n'
help_center/coupon_not_applied.html           | '[Meu cupom de desconto não funcionou | Central de Ajuda TechNova]\nMeu cupom de d'
help_center/support_channels_and_hours.html   | '[Canais e horários de atendimento | Central de Ajuda TechNova]\nO que ter em mãos'
faq.csv                                       | '[A bateria do meu fone está durando menos. Isso é coberto?]\nPergunta: A bateria '


**Observe:** canais de atendimento, cupons, bateria de fone. Nenhum trecho fala de vagas, mas todos chegariam ao `generate`
como se fossem contexto. O RAG simples tem três fraquezas aqui:

1. **Não confere a busca**: gera com o que veio. Quando os trechos são parecidos com a pergunta mas tratam de outra
   coisa (como o Fone Nova Sport na aula anterior), a resposta sai errada com cara de certa.
2. **Não tenta de novo**: se a consulta foi mal formulada, não há uma segunda busca com outras palavras.
3. **Não tem uma saída de segurança** decidida pelo código: quem decide "não sei" é o modelo, no meio da geração.

O **RAG corretivo** (Corrective RAG, ou CRAG) acrescenta um passo de verificação entre a busca e a geração, e deixa o
grafo decidir o que fazer com o resultado. Vamos construir cada peça.

## 2. Avaliar a relevância: o nó `grade`

O `grade` recebe a pergunta e os trechos numerados e devolve **quais números ajudam de fato**. Usamos a saída
estruturada do curso de LangChain com o schema `RelevanceGrade` (visto na aula anterior) e o `GRADE_PROMPT`, de
[src/prompts.py](../src/prompts.py).

In [4]:
from src.graph.corrective_rag import RelevanceGrade
from src.models import get_llm, token_usage
from src.prompts import GRADE_PROMPT
from src.rag.generation import format_context

print(GRADE_PROMPT.messages[0].prompt.template)

grader = GRADE_PROMPT | get_llm().with_structured_output(RelevanceGrade, method="function_calling", include_raw=True)


def grade_chunks(question: str, chunks: list) -> tuple[list, RelevanceGrade | None, dict]:
    """Uma chamada avalia todos os trechos. Devolve os aprovados, a avaliação e o consumo."""
    output = grader.invoke({"question": question, "context": format_context(chunks)})
    numbers = output["parsed"].relevant if output["parsed"] else []
    # O modelo pode repetir números ou inventar um que não existe: deduplicamos e validamos.
    relevant = [chunks[n - 1] for n in dict.fromkeys(numbers) if 1 <= n <= len(chunks)]
    return relevant, output["parsed"], token_usage(output["raw"])

Você filtra os trechos recuperados de uma base de conhecimento. Marque como relevantes SOMENTE os trechos que contêm informação útil para responder a pergunta.


No produto, o `retrieve` busca **dois trechos a mais** do que o `generate` vai usar (`settings.retrieval_k + 2`): a
avaliação descarta alguns, e sobra folga para manter até `retrieval_k` aprovados. Vamos fazer o mesmo.

In [5]:
from src.config import settings

for question in ["Em quantos dias posso trocar um produto?", job_question]:
    candidates = retriever.search(question, k=settings.retrieval_k + 2)
    relevant, grade, usage = grade_chunks(question, candidates)
    print(f"Pergunta: {question}")
    for number, chunk in enumerate(candidates, start=1):
        mark = "✔" if chunk in relevant else " "
        print(f"  {mark} [{number}] {origin(chunk)}")
    print(f"  justificativa: {grade.reasoning if grade else '(sem avaliação)'}")
    print(f"  consumo: {usage['input_tokens']} tokens de entrada, {usage['output_tokens']} de saída\n")

Pergunta: Em quantos dias posso trocar um produto?
  ✔ [1] faq.csv
  ✔ [2] help_center/exchange_for_another_model.html
  ✔ [3] faq.csv
  ✔ [4] policies/exchange.md
    [5] help_center/exchange_for_another_model.html
  ✔ [6] policies/exchange.md
  justificativa: Os trechos 1, 2, 3 e 6 informam diretamente o prazo de troca (30 dias corridos após a entrega). O trecho 4 traz um exemplo prático que confirma esse prazo. O trecho 5 fala do processo após a solicitação, não do prazo para trocar.
  consumo: 1215 tokens de entrada, 92 de saída



Pergunta: A TechNova está contratando? Onde mando meu currículo?
    [1] help_center/support_channels_and_hours.html
    [2] help_center/coupon_not_applied.html
    [3] help_center/support_channels_and_hours.html
    [4] faq.csv
    [5] help_center/support_channels_and_hours.html
    [6] help_center/track_order.html
  justificativa: Nenhum trecho fala sobre vagas, recrutamento ou envio de currículo; os trechos tratam de atendimento, contatos, cupons e rastreio.
  consumo: 1551 tokens de entrada, 53 de saída



**Observe:**

- Para a pergunta de troca, a avaliação aprova os trechos que falam de prazo e descarta os que só "parecem" com o assunto.
- Para a pergunta de vagas, **nenhum** trecho é aprovado. Esse é o sinal que o RAG simples não tinha.
- O consumo é quase todo de **entrada** (os trechos inteiros vão no prompt); a saída é curta, só números e uma frase.

> 💡 **Dica:** o artigo original do CRAG avalia **um trecho por chamada**. Aqui uma única chamada avalia todos: o custo
> cai de `k` chamadas para 1, e o modelo ainda compara os trechos entre si. Com muitos trechos longos, vale voltar a
> avaliar em lotes.

> ⚠️ O `grade` é um **filtro**, não uma garantia. Ele responde "este trecho ajuda?", não "a resposta está certa?". Um
> trecho sobre garantia pode ser aprovado para uma pergunta sobre garantia **estendida**, que a base não cobre. Voltamos
> a esse caso no fim da aula.

## 3. Decidir: a função `route`

Com a avaliação no estado, a decisão é **código**, não modelo. Como vimos no curso de LangGraph, a função da aresta
condicional só **lê** o estado e devolve o nome do próximo nó. O contador de tentativas é incrementado por um nó
(`rewrite`), nunca pela função de rota.

In [6]:
from typing import Literal

from langchain_core.documents import Document

MAX_REWRITES = settings.max_rewrites  # variável MAX_REWRITES no .env (padrão: 2)


def route(state: dict) -> Literal["generate", "rewrite", "no_answer"]:
    if state.get("relevant"):
        return "generate"
    if state.get("rewrites", 0) < MAX_REWRITES:
        return "rewrite"
    return "no_answer"


fake_chunk = Document("Trocas em até 30 dias.", metadata={"source": "policies/exchange.md"})
cases = [
    {"relevant": [fake_chunk], "rewrites": 0},
    {"relevant": [fake_chunk], "rewrites": 2},
    {"relevant": [], "rewrites": 0},
    {"relevant": [], "rewrites": 1},
    {"relevant": [], "rewrites": 2},
]
print(f"MAX_REWRITES = {MAX_REWRITES}\n")
print(f"{'relevantes':>10} | {'reescritas':>10} | próximo nó")
for case in cases:
    print(f"{len(case['relevant']):>10} | {case['rewrites']:>10} | {route(case)}")

MAX_REWRITES = 2

relevantes | reescritas | próximo nó
         1 |          0 | generate
         1 |          2 | generate
         0 |          0 | rewrite
         0 |          1 | rewrite
         0 |          2 | no_answer


Três saídas, três destinos: há trechos relevantes, **gera**; não há e ainda sobram tentativas, **reescreve**; não há e
o limite acabou, **desiste com segurança**. Repare que achar trechos relevantes tem prioridade sobre o contador: na
última tentativa, se a busca acertar, o grafo gera normalmente.

## 4. Reescrever a consulta: o nó `rewrite`

Quando nada é aprovado, pode ser que a base não tenha a resposta, ou que a **consulta** tenha sido mal formulada
(gíria, termo do cliente diferente do termo do documento). O `rewrite` pede ao modelo outra consulta, com palavras que
apareceriam em políticas e manuais.

In [7]:
from langchain_core.output_parsers import StrOutputParser

from src.prompts import REWRITE_PROMPT

print(REWRITE_PROMPT.messages[0].prompt.template, "\n")

rewriter = REWRITE_PROMPT | get_llm() | StrOutputParser()
queries = [job_question]
new_query = rewriter.invoke({"question": job_question, "tried": "; ".join(queries)}).strip()
print("Nova consulta:", new_query)
for chunk in retriever.search(new_query, k=settings.retrieval_k + 2):
    print(f"  {origin(chunk)}")

A busca na base de conhecimento da TechNova não encontrou trechos relevantes para a pergunta. Reescreva a consulta de busca com outras palavras: use termos que apareceriam em políticas, manuais ou artigos de ajuda, e remova detalhes irrelevantes. É uma busca na base interna, não na web: não use operadores como site:, aspas ou OR. Devolva apenas a nova consulta, em uma linha curta. 



Nova consulta: vagas de emprego TechNova enviar currículo candidatura carreira


  help_center/support_channels_and_hours.html
  manuals/cx-001_nova_boom.pdf p.5
  policies/privacy.md
  policies/warranty.md
  help_center/support_channels_and_hours.html
  manuals/nb-002_nova_pro_16.pdf p.5


Três cuidados no prompt e no nó:

- **Consultas já tentadas** vão no prompt (`tried`): sem isso, o modelo tende a devolver a mesma consulta de novo.
- **Sem operadores de busca web** (`site:`, aspas, `OR`): os modelos adoram escrevê-los, e o BM25 e o FAISS não os
  entendem.
- **Limite explícito**: cada reescrita custa uma chamada de reescrita, uma busca e uma nova avaliação. O limite fica em
  `settings.max_rewrites`, e o contador `rewrites` é zerado no início do turno (aula anterior).

A pergunta é a **mesma**; só a **consulta** muda. Por isso o estado tem `question` (usada no `grade` e no `generate`) e
`queries` (a lista de consultas tentadas; a última é a atual, usada no `retrieve`).

## 5. A resposta de segurança: o nó `no_answer`

Quando o limite acaba, o grafo responde com uma mensagem fixa, sem chamar o modelo:

In [8]:
from src.prompts import NO_ANSWER_MESSAGE

print(NO_ANSWER_MESSAGE)

Não encontrei essa informação nos documentos da TechNova. Se quiser, posso encaminhar sua dúvida para um atendente.


Por que uma mensagem fixa, e não pedir ao modelo para "dizer que não sabe"?

| | `generate` com trechos ruins | `no_answer` |
|---|---|---|
| Quem decide | o modelo, no meio da geração | o código, pela rota |
| Custo | uma chamada com todo o contexto | zero |
| Risco | responder com o que veio (alucinação apoiada em trecho errado) | nenhum |
| Mensagem | varia | sempre a mesma, com o encaminhamento para um atendente |

O `generate` do produto também pode concluir que não há resposta (`found=False`, quando o modelo decide que o contexto
não trata do assunto). As duas saídas usam a mesma mensagem, e o front-end não mostra fontes nesses casos.

## 6. Do notebook para o projeto: `build_rag_graph`

`build_rag_graph(retriever=None, checkpointer=None)`, em [src/graph/corrective_rag.py](../src/graph/corrective_rag.py),
monta o grafo com as peças desta aula e o `condense` da aula anterior. O `app.py` usa exatamente essa função, com um
`SqliteSaver` como checkpointer.

### 6.1 O desenho

`get_graph().draw_mermaid()` gera o diagrama a partir do grafo compilado: é a melhor documentação, porque nunca fica
desatualizada.

In [9]:
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

from src.graph.corrective_rag import build_rag_graph

graph = build_rag_graph(retriever, checkpointer=InMemorySaver())
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	condense(condense)
	retrieve(retrieve)
	grade(grade)
	rewrite(rewrite)
	generate(generate)
	no_answer(no_answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> condense;
	condense --> retrieve;
	grade -.-> generate;
	grade -.-> no_answer;
	grade -.-> rewrite;
	retrieve --> grade;
	rewrite --> retrieve;
	generate --> __end__;
	no_answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



As arestas tracejadas (`-.->`) são as condicionais, saindo de `grade`. O ciclo `rewrite -> retrieve -> grade` é o único
laço do grafo, e o contador garante que ele termina.

### 6.2 Cada caminho, passo a passo

A função `run_turn` executa um turno com `stream_mode="updates"`, imprime o que cada nó escreveu e calcula o consumo
**do turno** (a chave `usage` acumula a conversa inteira, então subtraímos o valor de antes). Três perguntas na **mesma
conversa**, uma para cada caminho: normal, continuação e sem resposta.

In [10]:
def summarize(node: str, values: dict) -> str:
    """O que cada nó escreveu, em uma linha."""
    values = values or {}
    if node == "condense":
        return f"question={values['question']!r} (e zera as chaves do turno)"
    if node == "retrieve":
        return f"chunks={[origin(c) for c in values['chunks']]}"
    if node == "grade":
        return f"relevant={[origin(c) for c in values['relevant']]}"
    if node == "rewrite":
        return f"rewrites={values['rewrites']}, nova consulta={values['queries'][-1]!r}"
    return f"found={values.get('found')}, sources={[s['source'] for s in values.get('sources', [])]}"


def run_turn(graph, text: str, thread_id: str) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    before = graph.get_state(config).values.get("usage") or {}
    path = []
    print(f"Cliente: {text}")
    for step in graph.stream({"messages": [HumanMessage(text)]}, config, stream_mode="updates"):
        for node, values in step.items():
            path.append(node)
            print(f"  [{node}] {summarize(node, values)}")
    state = graph.get_state(config).values
    usage = {key: value - before.get(key, 0) for key, value in state["usage"].items()}
    print(f"  Assistente: {state['messages'][-1].content}")
    print(f"  consumo do turno: {usage['calls']} chamadas ao modelo, {usage['total_tokens']} tokens\n")
    return {"question": text, "path": path, "usage": usage, "found": state["found"]}

In [11]:
runs = [
    run_turn(graph, "Em quantos dias posso trocar um produto?", "demo"),
    run_turn(graph, "E se ele veio com defeito?", "demo"),
    run_turn(graph, job_question, "demo"),
]

Cliente: Em quantos dias posso trocar um produto?
  [condense] question='Em quantos dias posso trocar um produto?' (e zera as chaves do turno)


  [retrieve] chunks=['faq.csv', 'help_center/exchange_for_another_model.html', 'faq.csv', 'policies/exchange.md', 'help_center/exchange_for_another_model.html', 'policies/exchange.md']


  [grade] relevant=['faq.csv', 'help_center/exchange_for_another_model.html', 'faq.csv', 'policies/exchange.md']


  [generate] found=True, sources=['faq.csv']
  Assistente: Você pode trocar um produto em até 30 dias corridos após a entrega. O produto deve estar sem sinais de uso, com todos os acessórios e na embalagem original. O frete da troca é gratuito. [1]
  consumo do turno: 2 chamadas ao modelo, 2353 tokens

Cliente: E se ele veio com defeito?


  [condense] question='Em quantos dias posso trocar um produto que veio com defeito?' (e zera as chaves do turno)


...

  [retrieve] chunks=['help_center/support_channels_and_hours.html', 'manuals/cx-001_nova_boom.pdf p.5', 'policies/privacy.md', 'policies/warranty.md', 'help_center/support_channels_and_hours.html', 'manuals/nb-002_nova_pro_16.pdf p.5']


  [grade] relevant=[]


  [rewrite] rewrites=2, nova consulta='processo seletivo TechNova enviar currículo candidatura emprego'


  [retrieve] chunks=['help_center/reset_password.html', 'policies/privacy.md', 'help_center/support_channels_and_hours.html', 'manuals/cx-001_nova_boom.pdf p.5', 'help_center/support_channels_and_hours.html', 'policies/warranty.md']


  [grade] relevant=[]
  [no_answer] found=False, sources=[]
  Assistente: Não encontrei essa informação nos documentos da TechNova. Se quiser, posso encaminhar sua dúvida para um atendente.
  consumo do turno: 6 chamadas ao modelo, 4709 tokens



**Observe:**

- **Normal**: `condense` (sem chamar o modelo, é a primeira pergunta), `retrieve`, `grade` e `generate`. Duas chamadas.
- **Continuação**: o `condense` reescreve "E se ele veio com defeito?" com o assunto da conversa (troca), e o resto é o
  caminho normal. Três chamadas.
- **Sem resposta**: duas reescritas, três buscas, três avaliações sem nenhum trecho aprovado, e o `no_answer` encerra
  com a mensagem de segurança. Repare que o `condense` também roda aqui (há histórico), mesmo a pergunta sendo
  independente.

Agora um caso de fronteira, em outra conversa: a base fala de garantia, mas não de garantia **estendida**.

In [12]:
runs.append(run_turn(graph, "Vocês vendem garantia estendida?", "edge"))

Cliente: Vocês vendem garantia estendida?
  [condense] question='Vocês vendem garantia estendida?' (e zera as chaves do turno)


  [retrieve] chunks=['policies/warranty.md', 'faq.csv', 'faq.csv', 'help_center/warranty_claim_step_by_step.html', 'faq.csv', 'manuals/nb-002_nova_pro_16.pdf p.5']


  [grade] relevant=[]


  [rewrite] rewrites=1, nova consulta='garantia estendida cobertura adicional compra proteção produto'


  [retrieve] chunks=['policies/warranty.md', 'help_center/warranty_claim_step_by_step.html', 'help_center/warranty_claim_step_by_step.html', 'policies/exchange.md', 'policies/warranty.md', 'policies/privacy.md']


  [grade] relevant=[]


  [rewrite] rewrites=2, nova consulta='garantia adicional proteção do produto compra cobertura assistência'


  [retrieve] chunks=['manuals/nb-002_nova_pro_16.pdf p.5', 'help_center/warranty_claim_step_by_step.html', 'help_center/warranty_claim_step_by_step.html', 'help_center/warranty_claim_step_by_step.html', 'policies/warranty.md', 'policies/warranty.md']


  [grade] relevant=[]
  [no_answer] found=False, sources=[]
  Assistente: Não encontrei essa informação nos documentos da TechNova. Se quiser, posso encaminhar sua dúvida para um atendente.
  consumo do turno: 5 chamadas ao modelo, 4313 tokens



**Observe** o caminho e a resposta. As três buscas trouxeram trechos sobre **garantia** (a política, o passo a passo
para acionar), e o `grade` rejeitou todos: nenhum fala de garantia **estendida**. Como é a primeira pergunta da conversa,
o `condense` não chamou o modelo, e o caminho sem resposta custou 5 chamadas, e não 6.

Esse é um caso de fronteira de verdade: trechos muito parecidos com a pergunta, sem a resposta. Em outras execuções, o
`grade` pode aprovar os trechos de garantia comum ("ajudam a entender o assunto"). Aí quem segura é o `generate`, que
conclui que o contexto não trata de garantia estendida e devolve `found=False`, com a mesma mensagem de segurança. É o
aviso da seção 2: o `grade` filtra trechos, não verifica respostas. Na aula 3.5 vamos
medir justamente isso: um LLM como juiz confere se cada **resposta** está apoiada nos trechos.

### 6.3 Custo x benefício

Quantas chamadas cada caminho custou, comparado ao RAG simples do módulo 02 (sempre uma chamada, a do `generate`)?

In [13]:
print(f"{'pergunta':<58} | {'chamadas':>8} | {'tokens':>6} | caminho")
print(f"{'RAG simples (qualquer pergunta)':<58} | {1:>8} | {'':>6} | retrieve -> generate")
for run in runs:
    print(f"{run['question'][:58]:<58} | {run['usage']['calls']:>8} | {run['usage']['total_tokens']:>6} | {' -> '.join(run['path'])}")

pergunta                                                   | chamadas | tokens | caminho
RAG simples (qualquer pergunta)                            |        1 |        | retrieve -> generate
Em quantos dias posso trocar um produto?                   |        2 |   2353 | condense -> retrieve -> grade -> generate
E se ele veio com defeito?                                 |        3 |   1954 | condense -> retrieve -> grade -> generate
A TechNova está contratando? Onde mando meu currículo?     |        6 |   4709 | condense -> retrieve -> grade -> rewrite -> retrieve -> grade -> rewrite -> retrieve -> grade -> no_answer
Vocês vendem garantia estendida?                           |        5 |   4313 | condense -> retrieve -> grade -> rewrite -> retrieve -> grade -> rewrite -> retrieve -> grade -> no_answer


Em geral, por caminho (com `MAX_REWRITES = 2`):

| Caminho | Chamadas ao modelo | Medido acima |
|---|---|---|
| Primeira pergunta, busca boa | `grade` + `generate` = 2 | 2 (troca) |
| Continuação, busca boa | `condense` + `grade` + `generate` = 3 | 3 (defeito) |
| Sem resposta, limite atingido | `condense` + 3 `grade` + 2 `rewrite` = 6 (5 na primeira pergunta) | 6 (vagas), 5 (garantia estendida) |
| Pior caso: duas reescritas e resposta | `condense` + 3 `grade` + 2 `rewrite` + `generate` = 7 | |

O que se ganha: menos respostas erradas com cara de certas (trechos irrelevantes não chegam ao `generate`), uma
segunda chance para consultas mal formuladas e uma saída de segurança decidida por código, sem custo.

Como controlar o custo:

- As chamadas extras têm saída curta (números, uma frase, uma consulta): um modelo menor para `condense`, `grade` e
  `rewrite` (`get_llm(model=...)`) costuma bastar; o `generate` fica com o modelo principal.
- `MAX_REWRITES` é o botão do pior caso: com 1, o caminho sem resposta cai de 6 para 4 chamadas.
- O caso comum (busca boa) custa 2 ou 3 chamadas; o pior caso aparece quando a base não sabe, justamente quando vale
  gastar um pouco mais para não inventar.

> 💡 **Dica:** meça antes de otimizar. As aulas 3.4 e 3.5 avaliam a recuperação (26 perguntas) e as respostas
> (10 perguntas): é essa avaliação que diz se o `grade` e o `rewrite` pagam o próprio custo na sua base.

## Resumo

- Um retriever sempre devolve `k` trechos; o RAG simples gera com eles mesmo quando nenhum serve.
- O nó `grade` avalia todos os trechos em uma chamada (`GRADE_PROMPT` + `RelevanceGrade`) e guarda os aprovados em
  `relevant`. É um filtro de trechos, não uma verificação da resposta.
- A função `route` decide por código: `generate` se há trechos relevantes, `rewrite` se ainda há tentativas,
  `no_answer` quando o limite (`settings.max_rewrites`) acaba.
- O `rewrite` usa o `REWRITE_PROMPT` com as consultas já tentadas; a pergunta (`question`) não muda, só a consulta
  (`queries`).
- `build_rag_graph` junta tudo com o `condense`; `stream_mode="updates"` mostra cada passo, e a chave `usage` mede o
  custo: de 2 chamadas (caminho normal) a 6 (sem resposta) ou 7 (continuação com duas reescritas e
  resposta), contra 1 do RAG simples.

## Próxima aula

Na aula 3.3 a busca deixa de ser uma etapa fixa e vira uma **ferramenta**: um agente decide se busca na base, se consulta um
pedido no banco da loja, se faz as duas coisas ou se responde direto.